<a href="https://colab.research.google.com/github/dave-heslop74/EMSC2010-W7-L1/blob/main/EMSC2010_W7_L1_NB2_uXXXXXXX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EMSC2010-W7-L1-NB2

### Estimating the posterior probability distribution for the proportion of ocean on a planet's surface

We'll apply Bayes theorem to find the posterior for a given prior and data set.

Import the libraries that we'll need.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize_scalar

The following function performs the analysis and creates a plot comparing the prior and the posterior. It also provides a summary of the posterior (which we will discuss).



In [ ]:
def plot_water_posterior(probes: int, ocean: int, threshold: float, alpha_prior: float = 1, beta_prior: float = 1):
    """
    Plot the prior and posterior distribution for the proportion of Earth's
    surface covered by water, given counts of random surface points.

    Uses a Beta-Binomial conjugate model:
        Prior:     p ~ Beta(alpha_prior, beta_prior)
        Likelihood: W | p ~ Binomial(n, p)
        Posterior: p | W ~ Beta(alpha_prior + W, beta_prior + L)

    Parameters
    ----------
    water : int
        Number of randomly selected points that landed on water.
    land : int
        Number of randomly selected points that landed on land.
    alpha_prior : float
        Alpha parameter of the Beta prior (default=1, uniform prior).
    beta_prior : float
        Beta parameter of the Beta prior (default=1, uniform prior).
    """
    water = ocean
    land = probes-ocean

    # --- Posterior parameters (conjugate update) ---
    alpha_post = alpha_prior + water
    beta_post  = beta_prior  + land

    # --- Posterior summary statistics ---
    posterior = stats.beta(alpha_post, beta_post)
    mean      = posterior.mean()
    hdi_lo, hdi_hi = beta_hdi(alpha_post, beta_post)
    n = water + land

    # --- Evaluate densities ---
    p = np.linspace(0, 1, 500)
    prior_pdf     = stats.beta(alpha_prior, beta_prior).pdf(p)
    posterior_pdf = posterior.pdf(p)
    map_estimate = (alpha_post - 1) / (alpha_post + beta_post - 2)

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(8, 4.5))

    ax.plot(p, prior_pdf,     color="grey",       lw=1.5, linestyle="--")
    ax.plot(p, posterior_pdf, color="steelblue",  lw=2.5)

    # 95% credible interval shading
    mask = (p >= hdi_lo) & (p <= hdi_hi)
    ax.fill_between(p[mask], posterior_pdf[mask], alpha=0.30, color="steelblue",
                    label=f"95% credible interval  [{hdi_lo:.3f}, {hdi_hi:.3f}]")

    # Posterior mean line
    ax.axvline(mean, color="steelblue", lw=1.5, linestyle=":",
               label=f"Posterior mean  {mean:.3f}")

    # Posterior MAP line
    ax.axvline(map_estimate, color="red", lw=1.5, linestyle=":",
               label=f"Maximum apriori probability  {map_estimate:.3f}")

    # Posterior MAP line
    ax.axvline(threshold, color="black", lw=1.5, linestyle=":",
               label=f"Threshold  {threshold:.3f}")



    ax.set_xlabel("Proportion of surface that is ocean (p)", fontsize=12)
    ax.set_ylabel("Probability density", fontsize=12)
    ax.set_title(
        f"Bayesian estimate of ocean coverage\n"
        f"n = {n} points  ({water} ocean, {land} land)",
        fontsize=13
    )
    ax.legend(fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_ylim(bottom=0)
    plt.tight_layout()
    plt.minorticks_on()
    plt.show()


    print(f"Posterior mean:          {mean:.4f}")
    print(f"Maximum apriori probability:          {map_estimate:.4f}")
    print(f"95% credible interval:   [{hdi_lo:.4f}, {hdi_hi:.4f}]")

    prob = posterior.sf(threshold)
    print(f"Probability(proportion of ocean > {threshold}) = {prob:.4f}")


def beta_hdi(alpha, beta, credibility=0.95):
    dist = stats.beta(alpha, beta)
    # For a given lower bound, the upper bound is fixed by the required mass
    def interval_width(lo):
        hi = dist.ppf(dist.cdf(lo) + credibility)
        return hi - lo
    result = minimize_scalar(interval_width, bounds=(0, dist.ppf(1 - credibility)),
                             method="bounded")
    lo = result.x
    hi = dist.ppf(dist.cdf(lo) + credibility)
    return lo, hi

Input your data and execute the function

In [ ]:
probes = 25 #total number of probes launched
ocean = 20 #number of probes that landed in the ocean
threshold = 0.75 #what is the proportion of ocean you are interested in.
plot_water_posterior(probes, ocean, threshold, alpha_prior = 1, beta_prior = 1)

Given the data and the prior, there is a 95% probability that the true proportion of ocean is within the 95% credible interval.

### Working with a different prior

For this specific problem, I've used a so-called *Beta* distribution to represent the prior. The distribution is controlled by two parameters:

* Larger α shifts the probability mass towards 1.
* Larger β shifts the probability mass towards 0.
* The ratio β/α determines where the bulk of the probability mass sits.
* The sum α + β controls the width of the distribution (i.e., how strongly held the belief is).

We initially used a uniform distribution as our prior which has:

* α = 1
* β = 1

However there could be good reasons to work with a different prior. For example, if we believe planets with large oceans are rare. We can control the shape of the prior distribution if we change ```alpha_prior``` and ```beta_prior``` in the function input.



In [ ]:
plot_water_posterior(probes, ocean, threshold, alpha_prior = 1,beta_prior = 5)

### Importantly the prior allows you to *encode* your pre-existing knowledge into the inference process.